In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# 1. Read source data
df_src = spark.read.table("sales.silver.customers")

# Remove rescued column if present
if "_rescued_data" in df_src.columns:
    df_src = df_src.drop("_rescued_data")

# Ensure one record per customer_id
df_src = df_src.dropDuplicates(["customer_id"])

target_table = "sales.gold.customers_new"

# 2. Check whether target table exists
if spark.catalog.tableExists(target_table):

    df_sink = spark.read.table(target_table)

    # Get maximum existing surrogate key
    max_value = (
        df_sink.agg(F.max("dim_customer_key").alias("max_key"))
        .first()["max_key"]
    )

    max_value = max_value if max_value is not None else 0

    # 3. Join source and target using customer_id
    df_join = (
        df_src.alias("s")
        .join(
            df_sink.alias("t"),
            F.col("s.customer_id") == F.col("t.customer_id"),
            "left"
        )
        .select(
            "s.*",
            F.col("t.dim_customer_key").alias("old_dim_customer_key"),
            F.col("t.create_date").alias("old_create_date")
        )
    )

    # 4. Existing customers
    df_old = (
        df_join
        .filter(F.col("old_dim_customer_key").isNotNull())
        .drop("old_dim_customer_key", "old_create_date")
        .join(
            df_sink.select(
                "customer_id",
                "dim_customer_key",
                "create_date"
            ),
            "customer_id",
            "inner"
        )
        .withColumn("update_date", F.current_timestamp())
    )

    # 5. New customers
    df_new = (
        df_join
        .filter(F.col("old_dim_customer_key").isNull())
        .drop("old_dim_customer_key", "old_create_date")
    )

    # Generate predictable, unique keys for new customers
    window_spec = Window.orderBy("customer_id")

    df_new = (
        df_new
        .withColumn(
            "dim_customer_key",
            F.row_number().over(window_spec) + F.lit(max_value)
        )
        .withColumn("create_date", F.current_timestamp())
        .withColumn("update_date", F.current_timestamp())
    )

    # 6. Combine existing and new customers
    df_final = df_old.unionByName(df_new)

    # 7. Merge using customer_id (business key)
    delta_table = DeltaTable.forName(spark, target_table)

    (
        delta_table.alias("t")
        .merge(
            df_final.alias("s"),
            "t.customer_id = s.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    # Initial load
    window_spec = Window.orderBy("customer_id")

    df_final = (
        df_src
        .withColumn(
            "dim_customer_key",
            F.row_number().over(window_spec)
        )
        .withColumn("create_date", F.current_timestamp())
        .withColumn("update_date", F.current_timestamp())
    )

    df_final.write.format("delta").mode("overwrite").saveAsTable(
        target_table
    )




In [0]:
# 8. Display final result
display(
    spark.read.table(target_table)
    .orderBy("dim_customer_key")
)